# 🎓 Tutorial 1: Generating Synthetic Patient Data

Before we can analyze clinical scores, we need patient data! In this project, we use a **Configuration-Driven Generator**.

Instead of writing Python loops to create fake patients, we define the clinical "rules" in a YAML file (`config/data_config.yaml`). The generator reads these rules to create realistic, messy EHR data.

## Step 1: Look at the YAML Configuration

Before we generate data, let's look at how we control the simulation using the `config/data_config.yaml` file. In professional clinical data science, we try to separate our **clinical logic** from our **Python code**. This allows doctors or clinical experts to adjust the rules of the pipeline without needing to know how to program!

In this configuration file, we define strict **physiological boundaries** for our synthetic patients. If we just used a standard random number generator, it might create a patient with a heart rate of 5,000 or an oxygen saturation of 150%, which is medically impossible.

Notice how we tell the system to constrain heart rates (`hr`) between a realistic 30 and 250 beats per minute, but strictly cap Oxygen Saturation (`o2sat`) at 100%.

You will also see a `prob` (probability) field. This mimics real-world hospital workflows! In a real ICU, a patient's heart rate is constantly monitored (`prob: 0.95`), but nurses might only check their temperature a few times a day (`prob: 0.33`). This tells our generator how often to record a measurement versus leaving it blank.

Open and read the `config/data_config.yaml` file.

Once you are done, run the cell bellow to see how we tell the system to make heart rates (`hr`) between 30 and 250 bpm, but Oxygen Saturation (`o2sat`) cannot exceed 100%!

In [5]:
import yaml
from pathlib import Path
import sys

project_root = Path().resolve().parent
sys.path.append(str(project_root))

# Let's load and print a small piece of our data config
config_path = project_root / 'config' / 'data_config.yaml'
with open(config_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("Number of patients to generate:", data_config['generation_params']['n_patients'])
print("\nHeart Rate Rules:\n", yaml.dump(data_config['ts_config']['hr']))
print("\nOxygen Saturation Rules:\n", yaml.dump(data_config['ts_config']['o2sat']))

Number of patients to generate: 100

Heart Rate Rules:
 prob: 0.95
range:
- 30.0
- 250.0
type: float
unit: bpm


Oxygen Saturation Rules:
 prob: 0.9
range:
- 50.0
- 100.0
type: float
unit: '%'



## Step 2: Generate the Data!

We import the `generate_clinical_data` function from our `src` folder. This outputs two tables:
1. **Static Data**: Demographics and history (e.g., Age, Gender).
2. **Time-Series Data**: Vitals and labs taken every 4 hours.

In [6]:
from src.generators import generate_clinical_data

df_static, df_ts = generate_clinical_data(
    config=data_config,
    n_patients=10, # Let's just generate 10 patients for this tutorial
    days=3,        # Over 3 days
    freq='4h'      # Readings every 4 hours
)

print("✅ Data Generated!")
print("Static Shape:", df_static.shape)
display(df_static.head(2))

print("\nTime-Series Shape:", df_ts.shape)
display(df_ts.head(3))

✅ Data Generated!
Static Shape: (10, 40)


,patient_id,age,sex,unit_type,admission_source,weight,bsi_source,microorganism,diabetes,hypertension,...,prior_abx_30d,chronic_dialysis,transfer_from_hosp,prior_esbl_history,hosp_last_90d,abx_last_90d,hosp_last_1y,prior_abx_90d,risk,sepsis_case
0,101,67,F,CCU,Transfer,149.1,Intra-abdominal,Other ESBL,0,1,...,0,0,0,0,0,1,0,0,33.5,0
1,102,36,M,General Floor,Transfer,138.0,Catheter,Other ESBL,0,0,...,0,0,0,0,0,0,0,0,18.0,0



Time-Series Shape: (3060, 5)


,patient_id,date,test,result,unit
0,101,2024-01-01,step,0.0,NaN
1,101,2024-01-01,onset_step,NaN,NaN
2,101,2024-01-01,is_sick,0.0,NaN


## Step 3: Making it "Messy" (Missingness)

Real hospital data is never perfect. Nurses get busy, and patients miss blood pressure readings. 

We apply a "Missingness Mask" to simulate real-world EHR dropouts.

In [7]:
from src.generators import apply_missingness

df_ts_missing = apply_missingness(df_ts, ts_config=data_config.get('ts_config', {}))

print("Notice the 'NaN' values that now appear in our data! This is what we have to clean later.")
display(df_ts_missing.head(5))

Notice the 'NaN' values that now appear in our data! This is what we have to clean later.


,patient_id,date,test,result,unit
0,101,2024-01-01,step,0.00,NaN
1,101,2024-01-01,onset_step,NaN,NaN
2,101,2024-01-01,is_sick,0.00,NaN
3,101,2024-01-01,hr,47.74,bpm
4,101,2024-01-01,rr,51.59,br/min



## Automating the Process

While it is crucial to understand how this configuration works, you don't have to run the generator manually line-by-line! We have completely automated this step in our project pipeline.

Whenever you update the rules in `data_config.yaml` (for example, if you want to simulate 10,000 patients instead of 100), you simply open your terminal and run:

```bash
python -m scripts.01_generate_data
```

This script will automatically read your YAML file, crunch the probabilities, apply the missingness masks, and save a brand new, ready-to-use dataset into a timestampedfolder inside data/synthetic/
